# QuixBugs Preprocessing — Bug-Fixing Task

Mirrors `preprocess_livecodebench.ipynb` so both datasets share the same core columns (`task_type`, `source_dataset`, `question_id`, `question_title`, `prompt`, `prompt_length_*`).

**Static only.** Everything uses `ast` / `difflib` / `tokenize` (parsing and comparing text). Nothing here executes the dataset's code, which keeps us inside the advisor's "static evaluation" guardrail.

In [156]:
import ast
import difflib
import io
import re
import textwrap
import tokenize
from pathlib import Path

import pandas as pd
from datasets import load_dataset

## STEP 1: Load QuixBugs from Hugging Face

In [157]:
url = "https://huggingface.co/datasets/Muennighoff/quixbugs/resolve/main/quixbugs_python.jsonl"
ds = load_dataset("json", data_files=url)

df_raw = ds["train"].to_pandas()
df = df_raw.copy()
df.head()

,name,buggy_program,docstring,solution,tests
0,bitcount,def bitcount(n):\n count = 0\n while n:\...,"""""""\nBitcount\nbitcount\n\n\nInput:\n n: a ...",def bitcount(n):\n count = 0\n while n:\...,assert bitcount(*[127]) == 7\nassert bitcount(...
1,breadth_first_search,from collections import deque as Queue\n\ndef ...,"""""""\nBreadth-First Search\n\n\nInput:\n sta...",from collections import deque as Queue\n\ndef ...,"class Node:\n def __init__(\n self,\..."
2,bucketsort,"def bucketsort(arr, k):\n counts = [0] * k\...","""""""\nBucket Sort\n\n\nInput:\n arr: A list ...","def bucketsort(arr, k):\n counts = [0] * k\...","assert bucketsort(*[[], 14]) == []\nassert buc..."
3,depth_first_search,"def depth_first_search(startnode, goalnode):\n...","""""""\nDepth-first Search\n\n\nInput:\n start...","def depth_first_search(startnode, goalnode):\n...","class Node:\n def __init__(\n self,\..."
4,detect_cycle,def detect_cycle(node):\n hare = tortoise =...,"""""""\nLinked List Cycle Detection\ntortoise-har...",def detect_cycle(node):\n hare = tortoise =...,"class Node:\n def __init__(\n self,\..."


In [158]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   name           40 non-null     str  
 1   buggy_program  40 non-null     str  
 2   docstring      40 non-null     str  
 3   solution       40 non-null     str  
 4   tests          40 non-null     str  
dtypes: str(5)
memory usage: 87.2 KB


## STEP 2: Check for Empty Strings & Duplicates

There are no nulls in this dataset, so the only thing to check is empty / whitespace-only strings. Any row with an empty field can't be turned into a scorable task, so it would be dropped. We also de-duplicate on `name` (the QuixBugs program id).

In [159]:
TEXT_COLS = ["name", "buggy_program", "docstring", "solution", "tests"]

empty = df[TEXT_COLS].apply(lambda col: col.str.strip().eq(""))
print("Empty strings per column:")
print(empty.sum())

df = df[~empty.any(axis=1)].reset_index(drop=True)
print(f"\nRemaining after dropping empty rows: {len(df)}")

Empty strings per column:
name             0
buggy_program    0
docstring        0
solution         0
tests            0
dtype: int64

Remaining after dropping empty rows: 40


In [160]:
before = len(df)
df = df.drop_duplicates(subset="name").reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate name rows")
print(f"Remaining tasks: {len(df)}")

Dropped 0 duplicate name rows
Remaining tasks: 40


## STEP 3: Standardize & Validate the Code

### 3a. Normalize text
Line endings → `\n`, trailing whitespace stripped, everything cast to plain `str`. Applied identically to buggy and fixed code, so exact-match / AST-diff comparisons stay fair.

In [161]:
def normalize_text(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    return "\n".join(line.rstrip() for line in s.split("\n")).strip("\n")


for col in ["buggy_program", "docstring", "solution", "tests"]:
    df[col] = df[col].astype(str).map(normalize_text)
df["name"] = df["name"].astype(str).str.strip()

### 3b. Check that the code parses
The buggy code, reference solution and tests must all be valid Python (checked with `ast.parse`, no execution).

In [162]:
def parses(src: str) -> bool:
    try:
        ast.parse(src)
        return True
    except SyntaxError:
        return False


valid = df[["buggy_program", "solution", "tests"]].map(parses).all(axis=1)
print(f"Dropped {(~valid).sum()} rows with code that doesn't parse")
df = df[valid].reset_index(drop=True)

Dropped 0 rows with code that doesn't parse


### 3c. Drop "no-op" bugs
If the buggy and fixed code have identical syntax trees (they differ only in formatting or comments), there is no bug to fix.

In [163]:
def ast_key(src: str) -> str:
    return ast.dump(ast.parse(src))


no_op = df["buggy_program"].map(ast_key) == df["solution"].map(ast_key)
print(f"Dropped {no_op.sum()} rows where buggy and fixed code are AST-identical")
df = df[~no_op].reset_index(drop=True)
print(f"Remaining tasks: {len(df)}")

Dropped 0 rows where buggy and fixed code are AST-identical
Remaining tasks: 40


## STEP 4: Parse the Tests & Docstring

QuixBugs `tests` comes in **two shapes**:

| `test_style` | What it looks like | Programs |
|---|---|---|
| `assert_list` | flat top-level lines: `assert f(*[...]) == ...` | 31 of 40 |
| `test_functions` | optional setup (`class Node`, fixtures), then `def test1(): ... assert ...` and `test1()` calls at the bottom | 9 of 40 (graph / linked-list / tree problems) |

One **test case** = one top-level `assert`, or one `def testN()` function.

### 4a. Split tests into setup + test cases

In [164]:
def is_test_fn(node) -> bool:
    return isinstance(node, ast.FunctionDef) and node.name.startswith("test")


def split_tests(src: str):
    """Return (setup_code, test_cases). Runner calls like `test1()` are dropped."""
    setup, cases = [], []
    for node in ast.parse(src).body:
        seg = ast.get_source_segment(src, node)
        if isinstance(node, ast.Assert) or is_test_fn(node):
            cases.append(seg)
        elif (isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
              and isinstance(node.value.func, ast.Name) and node.value.func.id.startswith("test")):
            continue
        else:
            setup.append(seg)
    return "\n\n".join(setup), cases


parsed = df["tests"].map(split_tests)
df["setup_code"] = parsed.map(lambda t: t[0])
df["test_cases"] = parsed.map(lambda t: t[1])
df["num_test_cases"] = df["test_cases"].map(len)
df["num_asserts"] = df["tests"].map(lambda s: sum(isinstance(n, ast.Assert) for n in ast.walk(ast.parse(s))))
df["test_style"] = df["test_cases"].map(lambda cs: "test_functions" if cs[0].startswith("def ") else "assert_list")

# Regression guard: every task needs at least one parsed test
empty_tests = df.loc[(df["num_test_cases"] == 0) | (df["num_asserts"] == 0), "name"].tolist()
assert not empty_tests, f"Tasks with no parsed tests: {empty_tests}"

print(df["test_style"].value_counts().to_string())
print(f"Tasks with setup code: {(df['setup_code'] != '').sum()}")

test_style
assert_list       31
test_functions     9
Tasks with setup code: 6


`num_asserts` counts *every* `assert`, including ones nested inside `testN()` functions. Use it (not `num_test_cases`) when comparing across the two styles.

### 4b. Flag order-dependent tests
If a `testN()` function mutates a module-level object (e.g. `node1.successor = ...` in `detect_cycle`), its tests depend on running in order and can't be scored one at a time.

In [165]:
def shares_state(src: str) -> bool:
    tree = ast.parse(src)
    module_vars = {t.id for n in tree.body if isinstance(n, ast.Assign) for t in n.targets if isinstance(t, ast.Name)}
    for fn in filter(is_test_fn, tree.body):
        local = {t.id for n in ast.walk(fn) if isinstance(n, ast.Assign) for t in n.targets if isinstance(t, ast.Name)}
        for n in ast.walk(fn):
            if isinstance(n, ast.Assign):
                for t in n.targets:
                    if isinstance(t, ast.Attribute) and isinstance(t.value, ast.Name) and t.value.id in module_vars - local:
                        return True
    return False


df["tests_share_state"] = df["tests"].map(shares_state)
print("Tests share mutable state:", df.loc[df["tests_share_state"], "name"].tolist())

Tests share mutable state: ['detect_cycle']


### 4c. Entry point
The function the tests call. Usually the same as `name`; if the solution has helper functions, `name` is still the one we want.

In [166]:
def get_entry_point(solution: str, name: str) -> str:
    defs = [n.name for n in ast.parse(solution).body if isinstance(n, ast.FunctionDef)]
    return name if name in defs else defs[-1]


df["entry_point"] = [get_entry_point(s, n) for s, n in zip(df["solution"], df["name"])]
print(f"Entry point differs from task name: {(df['entry_point'] != df['name']).sum()} rows")

Entry point differs from task name: 0 rows


### 4d. Clean the docstring (the problem statement)

The docstring is the spec the model gets. Cleaning does three things:
1. strips the surrounding `"""` and common indentation;
2. **removes the duplicate title.** The first line is the problem title; in 18 of 40 docstrings the next line repeats it as a lowercase slug (`Bitcount` / `bitcount`, `Making Change` / `change`, `Shortest Path` / `dijkstra`). That slug line is dropped so the title appears once;
3. collapses runs of 3+ blank lines into one blank line.

In [167]:
SLUG = re.compile(r"[a-z0-9_-]+")


def clean_docstring(s: str) -> str:
    s = s.strip()
    if s.startswith('"""') and s.endswith('"""'):
        s = s[3:-3]
    lines = textwrap.dedent(s).strip().split("\n")

    # line 0 is the title; drop the next non-empty line if it is only a slug
    nxt = next((i for i, line in enumerate(lines[1:], 1) if line.strip()), None)
    if nxt is not None and SLUG.fullmatch(lines[nxt].strip()):
        del lines[nxt]

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines))


df["docstring_clean"] = df["docstring"].map(clean_docstring)
print("BEFORE:\n" + df.loc[df["name"] == "bitcount", "docstring"].iloc[0])
print("\nAFTER:\n" + df.loc[df["name"] == "bitcount", "docstring_clean"].iloc[0])

BEFORE:
"""
Bitcount
bitcount


Input:
    n: a nonnegative int

Output:
    The number of 1-bits in the binary encoding of n

Examples:
    >>> bitcount(127)
    7
    >>> bitcount(128)
    1
"""

AFTER:
Bitcount

Input:
    n: a nonnegative int

Output:
    The number of 1-bits in the binary encoding of n

Examples:
    >>> bitcount(127)
    7
    >>> bitcount(128)
    1


## STEP 5: Bug-Diff Features

The buggy → fixed diff is the ground truth for the bug-fix task, and it's useful for the Evaluator rubric later (e.g. "did the model touch the same lines as the reference fix?"). Computed statically with `difflib`, **ignoring comments and blank lines** (a few buggy programs carry a comment the reference solution doesn't, e.g. `# Python 3` in `possible_change`, and that shouldn't count as part of the bug).

In [168]:
def code_lines(src: str) -> list:
    """Code lines with comments and blank lines removed."""
    lines = src.split("\n")
    for tok in tokenize.generate_tokens(io.StringIO(src).readline):
        if tok.type == tokenize.COMMENT:
            row, col = tok.start
            lines[row - 1] = lines[row - 1][:col]
    return [ln.rstrip() for ln in lines if ln.strip()]


def diff_features(buggy: str, fixed: str) -> pd.Series:
    b, f = code_lines(buggy), code_lines(fixed)
    hunks = changed_b = changed_f = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, b, f).get_opcodes():
        if tag != "equal":
            hunks += 1
            changed_b += i2 - i1
            changed_f += j2 - j1
    return pd.Series(
        {
            "num_changed_lines": max(changed_b, changed_f),
            "num_diff_hunks": hunks,
            "char_similarity": round(difflib.SequenceMatcher(None, "\n".join(b), "\n".join(f)).ratio(), 4),
            "bug_diff": "\n".join(difflib.unified_diff(b, f, "buggy", "fixed", lineterm="", n=0)),
        }
    )


diffs = df.apply(lambda r: diff_features(r["buggy_program"], r["solution"]), axis=1)
df = pd.concat([df, diffs], axis=1)
df["num_changed_lines"] = df["num_changed_lines"].astype(int)
df["num_diff_hunks"] = df["num_diff_hunks"].astype(int)

print(df["num_changed_lines"].value_counts().sort_index().to_string())
df[["name", "num_changed_lines", "num_diff_hunks", "char_similarity"]].head()

num_changed_lines
1    40


,name,num_changed_lines,num_diff_hunks,char_similarity
0,bitcount,1,1,0.9898
1,breadth_first_search,1,1,0.7789
2,bucketsort,1,1,0.9779
3,depth_first_search,1,1,0.9542
4,detect_cycle,1,1,0.9321


## STEP 6: Build the Model-Ready Prompt

A short fixed instruction, the spec (cleaned docstring, title once), then the buggy code. We don't say how many bugs there are or where. `BUGFIX_INSTRUCTION` is a constant so the Router agent's specialised prompts can swap it later without touching the data.

In [169]:
BUGFIX_INSTRUCTION = (
    "Fix the bug in the following Python function. "
    "Return the complete corrected function (plus any helper code it needs) "
    "in a single ```python code block. Do not change the function signature."
)


def build_prompt(row) -> str:
    return (
        f"{BUGFIX_INSTRUCTION}\n\n"
        f"Specification:\n{row['docstring_clean']}\n\n"
        f"Buggy code:\n```python\n{row['buggy_program']}\n```"
    )


df["prompt"] = df.apply(build_prompt, axis=1)
df["prompt_length_chars"] = df["prompt"].str.len()
df["prompt_length_words"] = df["prompt"].str.split().map(len)
print(df["prompt"].iloc[0])

Fix the bug in the following Python function. Return the complete corrected function (plus any helper code it needs) in a single ```python code block. Do not change the function signature.

Specification:
Bitcount

Input:
    n: a nonnegative int

Output:
    The number of 1-bits in the binary encoding of n

Examples:
    >>> bitcount(127)
    7
    >>> bitcount(128)
    1

Buggy code:
```python
def bitcount(n):
    count = 0
    while n:
        n ^= n - 1
        count += 1
    return count
```


## STEP 7: Tag Task Type & Finalize Schema

Shared with the LiveCodeBench output: `task_type`, `source_dataset`, `question_id`, `question_title`, `prompt`, `prompt_length_chars`, `prompt_length_words`. LiveCodeBench-only fields (`platform`, `contest_*`, `difficulty`, public/private split) have no QuixBugs equivalent and are not invented here.

In [170]:
df["task_type"] = "bug_fixing"
df["source_dataset"] = "quixbugs"
df["language"] = "python"
df["question_id"] = "quixbugs_" + df["name"]
df["question_title"] = df["name"]

df = df.rename(
    columns={
        "buggy_program": "buggy_code",
        "solution": "reference_solution",
        "tests": "test_code",
    }
)

final_columns = [
    "task_type", "source_dataset", "question_id", "question_title", "entry_point",
    "prompt",
    "docstring_clean", "buggy_code", "reference_solution",
    "bug_diff", "num_changed_lines",
    "setup_code", "test_cases", "tests_share_state", "test_code",
]
df_final = df[final_columns].rename(columns={"docstring_clean": "docstring"}).copy()
df_final.head()

,task_type,source_dataset,question_id,question_title,entry_point,prompt,docstring,buggy_code,reference_solution,bug_diff,num_changed_lines,setup_code,test_cases,tests_share_state,test_code
0,bug_fixing,quixbugs,quixbugs_bitcount,bitcount,bitcount,Fix the bug in the following Python function. ...,Bitcount\n\nInput:\n n: a nonnegative int\n...,def bitcount(n):\n count = 0\n while n:\...,def bitcount(n):\n count = 0\n while n:\...,--- buggy\n+++ fixed\n@@ -4 +4 @@\n- n ...,1,,"[assert bitcount(*[127]) == 7, assert bitcount...",False,assert bitcount(*[127]) == 7\nassert bitcount(...
1,bug_fixing,quixbugs,quixbugs_breadth_first_search,breadth_first_search,breadth_first_search,Fix the bug in the following Python function. ...,Breadth-First Search\n\nInput:\n startnode:...,from collections import deque as Queue\n\ndef ...,from collections import deque as Queue\n\ndef ...,--- buggy\n+++ fixed\n@@ -7 +7 @@\n- while ...,1,"class Node:\n def __init__(\n self,\...","[def test1():\n """"""Case 1: Strongly connect...",False,"class Node:\n def __init__(\n self,\..."
2,bug_fixing,quixbugs,quixbugs_bucketsort,bucketsort,bucketsort,Fix the bug in the following Python function. ...,Bucket Sort\n\nInput:\n arr: A list of smal...,"def bucketsort(arr, k):\n counts = [0] * k\...","def bucketsort(arr, k):\n counts = [0] * k\...","--- buggy\n+++ fixed\n@@ -6 +6 @@\n- for i,...",1,,"[assert bucketsort(*[[], 14]) == [], assert bu...",False,"assert bucketsort(*[[], 14]) == []\nassert buc..."
3,bug_fixing,quixbugs,quixbugs_depth_first_search,depth_first_search,depth_first_search,Fix the bug in the following Python function. ...,Depth-first Search\n\nInput:\n startnode: A...,"def depth_first_search(startnode, goalnode):\n...","def depth_first_search(startnode, goalnode):\n...","--- buggy\n+++ fixed\n@@ -8,0 +9 @@\n+ ...",1,"class Node:\n def __init__(\n self,\...","[def test1():\n """"""Case 1: Strongly connect...",False,"class Node:\n def __init__(\n self,\..."
4,bug_fixing,quixbugs,quixbugs_detect_cycle,detect_cycle,detect_cycle,Fix the bug in the following Python function. ...,Linked List Cycle Detection\n\nImplements the ...,def detect_cycle(node):\n hare = tortoise =...,def detect_cycle(node):\n hare = tortoise =...,--- buggy\n+++ fixed\n@@ -4 +4 @@\n- if...,1,"class Node:\n def __init__(\n self,\...","[def test1():\n """"""Case 1: 5-node list inpu...",True,"class Node:\n def __init__(\n self,\..."


## STEP 8: Save the Processed Dataset

JSONL (the `test_cases` column is a nested list). We read it back to confirm the list survives.

In [171]:
output_dir = Path("../data/bug_fixes/quixbugs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "quixbugs_processed.jsonl"
df_final.to_json(output_path, orient="records", lines=True)
print(f"Saved {len(df_final)} processed tasks to {output_path}")

check = pd.read_json(output_path, lines=True, dtype=False)
assert len(check) == len(df_final) and isinstance(check.loc[0, "test_cases"], list)
print("Round-trip check passed")

Saved 40 processed tasks to ../data/bug_fixes/quixbugs/quixbugs_processed.jsonl
Round-trip check passed


## STEP 9: Final Check — Ready for EDA / Baseline Testing

In [172]:
print(f"Final dataset: {len(df_final)} tasks, {df_final['question_id'].nunique()} unique problems")
print(df_final[["num_test_cases", "num_asserts", "num_changed_lines", "prompt_length_words"]].describe().to_string())

sample = df_final.iloc[0]
print("\n--- Sample prompt that would be sent to a baseline model ---\n")
print(sample["prompt"])
print("\n--- Reference diff (ground truth) ---\n")
print(sample["bug_diff"])

Final dataset: 40 tasks, 40 unique problems


KeyError: "['num_test_cases', 'num_asserts', 'prompt_length_words'] not in index"